# 04 Model Evaluation

This notebook evaluates the selected leakage-aware Random Forest model and translates model results into stakeholder-ready recommendations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)

In [ ]:
from evaluate import compare_models, plot_confusion_matrix, plot_roc_curve
from feature_engineering import add_retention_features
from train import prepare_train_test_data, train_candidate_models
from utils import load_processed_data, save_plot

clean_df = load_processed_data()
df = add_retention_features(clean_df)

X_train, X_test, y_train, y_test = prepare_train_test_data(df, leakage_aware=True)
models = train_candidate_models(X_train, y_train)
rf_model = models["Random Forest"]
model_comparison = compare_models(models, X_test, y_test)
model_comparison

## Confusion Matrix

The confusion matrix shows how many employees are correctly and incorrectly classified. In this HR use case, false negatives are especially important because they represent employees who may leave without being flagged for support.

In [ ]:
fig, ax = plot_confusion_matrix(rf_model, X_test, y_test, title="Random Forest Confusion Matrix")
save_plot(fig, "images/results/random_forest_confusion_matrix.png")
plt.show()

## ROC Curve

The ROC curve summarizes how well the model separates employees who left from employees who stayed across classification thresholds.

In [ ]:
fig, ax = plot_roc_curve(rf_model, X_test, y_test, title="Random Forest ROC Curve")
save_plot(fig, "images/results/roc_curve.png")
plt.show()

## Business Interpretation

The model identifies employees with elevated attrition risk, but the prediction should be used as a decision-support signal. It should trigger workload review, manager check-ins, career development conversations, and policy analysis.

It should not be used to punish employees, reduce opportunities, or make employment decisions automatically.

## Recommended HR Actions

- Review employees assigned to unusually high project counts before workload becomes unsustainable.
- Audit teams where high evaluation scores coincide with consistently high monthly hours.
- Investigate the four-year tenure group for promotion bottlenecks, role stagnation, or unmet career expectations.
- Create targeted retention check-ins for high-risk roles or departments instead of broad generic surveys.
- Track whether salary bands and promotion history create avoidable attrition risk across departments.